# Scoring
In this notebook, we score the BART-filtered headlines for "jitteriness".

First we load the labeled headlines and select only those labeled "relevant" by BART.

In [1]:
import pandas as pd

df = pd.read_csv("data/labeled_headlines.csv")
df = df[df.relevant == 1]
df.dropna(inplace=True)

Then we load the model and define the jitteriness category:

In [2]:
from transformers import pipeline
import torch

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.mps.is_available() else "cpu")
)

print("Device:", device)

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
    dtype=torch.float16,
)

candidate_labels = ["jittery", "non-jittery"]

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

We also define the batch processing function used to extract the "jittery" score:

In [3]:
def scorer(batch):
    results = classifier(batch["concat"], candidate_labels=candidate_labels)

    scores = [(dict(zip(res["labels"], res["scores"]))["jittery"]) for res in results]

    return {"score": scores}

Finally, we score each headline

In [4]:
from datasets import Dataset

ds = Dataset.from_pandas(df[["concat"]])
with torch.inference_mode():
    ds = ds.map(
        scorer,
        batched=True,
        batch_size=128,
    )

df["score"] = ds["score"]

Map:   0%|          | 0/973 [00:00<?, ? examples/s]

and save the scored dataset

In [5]:
df.to_csv("data/scored_headlines.csv", index=False)

It looks like reasonable training thresholds are `>0.8` for "jittery" and `<0.4` for "non-jittery":

In [7]:
df[(df.score > 0.8) | (df.score < 0.5)]

,title,description,timestamp,concat,relevant,score
5,Trump Is in China as Iran War Continues With N...,"Before leaving Washington on Tuesday, the pres...","time.struct_time(tm_year=2026, tm_mon=5, tm_md...",Trump Is in China as Iran War Continues With N...,1,0.895614
8,A Tech Tycoon’s Prosecution Raises Fears of Au...,Nadiem Makarim founded a popular app before jo...,"time.struct_time(tm_year=2026, tm_mon=5, tm_md...",A Tech Tycoon’s Prosecution Raises Fears of Au...,1,0.978062
13,Why A.I. is the Hidden Minefield of Trump’s Ch...,The leaders of both countries are expected to ...,"time.struct_time(tm_year=2026, tm_mon=5, tm_md...",Why A.I. is the Hidden Minefield of Trump’s Ch...,1,0.920361
16,"Chinese Firms Plot Secret Arms Sales to Iran, ...",The effort involves plans to send weapons thro...,"time.struct_time(tm_year=2026, tm_mon=5, tm_md...","Chinese Firms Plot Secret Arms Sales to Iran, ...",1,0.887644
24,Trump Tower Deal in Australia Falls Apart And ...,The property developer says the war in Iran an...,"time.struct_time(tm_year=2026, tm_mon=5, tm_md...",Trump Tower Deal in Australia Falls Apart And ...,1,0.958382
...,...,...,...,...,...,...
3449,TSA Experiments With Off-Site Screening to Rel...,A pilot program for Delta and JetBlue passenge...,"time.struct_time(tm_year=2026, tm_mon=5, tm_md...",TSA Experiments With Off-Site Screening to Rel...,1,0.453262
3450,The Academic Scramble to Prepare Future Accoun...,"The school year is ending, but for accounting ...","time.struct_time(tm_year=2026, tm_mon=5, tm_md...",The Academic Scramble to Prepare Future Accoun...,1,0.951886
3478,"TrumpRx Adds Generic Drugs, With Mark Cuban, G...",President Trump announced the addition of 600 ...,"time.struct_time(tm_year=2026, tm_mon=5, tm_md...","TrumpRx Adds Generic Drugs, With Mark Cuban, G...",1,0.436082
3481,How a Hantavirus Outbreak Turned a Nature Crui...,The hantavirus outbreak on the MV Hondius set ...,"time.struct_time(tm_year=2026, tm_mon=5, tm_md...",How a Hantavirus Outbreak Turned a Nature Crui...,1,0.985100
